
# Download de materiais — Moodle Acadêmico UFRGS

Este notebook entra no curso **45959 - Métodos Clássicos de Aprendizagem de Máquina** e baixa os arquivos de material disponibilizados ao seu usuário.

**Segurança:** as credenciais são solicitadas durante a execução, sem serem gravadas no notebook. Não preencha senha em células de código.

**Uso responsável:** baixe somente conteúdos aos quais você já possui acesso e respeite as regras do Moodle/UFRGS.


In [ ]:
# Execute somente uma vez, caso as bibliotecas ainda não estejam instaladas.
#%pip install -q requests beautifulsoup4 tqdm
#python -m pip install requests beautifulsoup4 tqdm #executar no terminal caso o comando acima não funcione

In [9]:

from __future__ import annotations

import getpass
import mimetypes
import re
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse, unquote

import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

# --- Curso e pasta de destino ---
COURSE_URL = "https://moodle.ufrgs.br/course/view.php?id=158151"
BASE_URL = "https://moodle.ufrgs.br/"
DESTINO = Path.home() / "Downloads" / "UFRGS_45959_Metodos_Classicos_Aprendizagem_de_Maquina"

# Para primeiro teste, deixe True. O notebook apenas lista os arquivos encontrados.
# Depois mude para False para efetuar os downloads.
SOMENTE_LISTAR = False

TIMEOUT = 60
PAUSA_ENTRE_DOWNLOADS_S = 0.25


In [10]:

def obter_formulario_login(sessao: requests.Session) -> tuple[str, dict]:
    """Abre a página de login e devolve a URL de envio e campos ocultos."""
    resposta = sessao.get(
        urljoin(BASE_URL, "login/index.php"),
        params={"wantsurl": COURSE_URL},
        timeout=TIMEOUT,
    )
    resposta.raise_for_status()
    sopa = BeautifulSoup(resposta.text, "html.parser")

    formulario = sopa.find("form", {"id": "login"}) or sopa.find("form")
    if formulario is None:
        raise RuntimeError("Não foi possível localizar o formulário de login do Moodle.")

    campos = {}
    for campo in formulario.select("input[name]"):
        campos[campo["name"]] = campo.get("value", "")

    acao = urljoin(resposta.url, formulario.get("action", ""))
    return acao, campos


def fazer_login(sessao: requests.Session, usuario: str, senha: str) -> None:
    acao, campos = obter_formulario_login(sessao)

    # Campos normalmente usados no Moodle. Mantém tokens ocultos, como logintoken.
    campos["username"] = usuario
    campos["password"] = senha

    resposta = sessao.post(
        acao,
        data=campos,
        timeout=TIMEOUT,
        allow_redirects=True,
    )
    resposta.raise_for_status()

    # Verificação genérica: se ainda há formulário de login, a autenticação falhou.
    sopa = BeautifulSoup(resposta.text, "html.parser")
    ainda_no_login = (
        "login/index.php" in resposta.url
        or sopa.find("input", {"name": "password"}) is not None
    )
    if ainda_no_login:
        raise RuntimeError(
            "Login não confirmado. Verifique o número do cartão, a senha ou "
            "se o Moodle apresentou alguma confirmação adicional."
        )


def nome_seguro(nome: str, fallback: str = "arquivo") -> str:
    nome = unquote(nome).strip()
    nome = re.sub(r'[<>:"/\\\\|?*\\x00-\\x1f]', "_", nome)
    nome = re.sub(r"\s+", " ", nome)
    return (nome[:180] or fallback).strip(" .")


def nome_do_arquivo(resposta: requests.Response, url: str, indice: int) -> str:
    """Prioriza Content-Disposition e depois usa a última parte da URL."""
    cd = resposta.headers.get("Content-Disposition", "")
    match = re.search(r"filename\\*?=(?:UTF-8''|\"?)([^\";]+)", cd, flags=re.I)
    if match:
        return nome_seguro(match.group(1), f"arquivo_{indice:03d}")

    caminho = unquote(urlparse(resposta.url).path)
    candidato = Path(caminho).name
    if candidato and "." in candidato and candidato.lower() not in {"pluginfile.php", "view.php"}:
        return nome_seguro(candidato, f"arquivo_{indice:03d}")

    tipo = resposta.headers.get("Content-Type", "").split(";")[0].strip()
    extensao = mimetypes.guess_extension(tipo) or ""
    return f"arquivo_{indice:03d}{extensao}"


def parece_arquivo(url: str) -> bool:
    """Reconhece os formatos de links mais comuns em Moodle."""
    caminho = urlparse(url).path.lower()
    return (
        "/pluginfile.php/" in caminho
        or "/mod/resource/view.php" in caminho
        or "/mod/folder/view.php" in caminho
    )


def links_do_html(html: str, pagina_url: str) -> set[str]:
    sopa = BeautifulSoup(html, "html.parser")
    encontrados = set()
    for a in sopa.select("a[href]"):
        href = urljoin(pagina_url, a["href"])
        url = urlparse(href)
        if url.scheme in {"http", "https"} and url.netloc == urlparse(BASE_URL).netloc:
            encontrados.add(href)
    return encontrados


def coletar_links_de_materiais(sessao: requests.Session, course_url: str) -> list[str]:
    """Lê a página do curso e expande pastas para encontrar arquivos internos."""
    resposta = sessao.get(course_url, timeout=TIMEOUT)
    resposta.raise_for_status()

    candidatos = links_do_html(resposta.text, resposta.url)
    fila_pastas = [u for u in candidatos if "/mod/folder/view.php" in urlparse(u).path.lower()]
    materiais = {
        u for u in candidatos
        if "/pluginfile.php/" in urlparse(u).path.lower()
        or "/mod/resource/view.php" in urlparse(u).path.lower()
    }

    visitadas = set()
    while fila_pastas:
        pasta = fila_pastas.pop()
        if pasta in visitadas:
            continue
        visitadas.add(pasta)

        r_pasta = sessao.get(pasta, timeout=TIMEOUT)
        r_pasta.raise_for_status()

        for link in links_do_html(r_pasta.text, r_pasta.url):
            caminho = urlparse(link).path.lower()
            if "/pluginfile.php/" in caminho or "/mod/resource/view.php" in caminho:
                materiais.add(link)
            elif "/mod/folder/view.php" in caminho and link not in visitadas:
                fila_pastas.append(link)

    return sorted(materiais)


def baixar_material(sessao: requests.Session, url: str, destino: Path, indice: int) -> Path | None:
    """Baixa um recurso. O Moodle redireciona links mod/resource para o arquivo real."""
    r = sessao.get(url, timeout=TIMEOUT, stream=True, allow_redirects=True)
    r.raise_for_status()

    content_type = r.headers.get("Content-Type", "").lower()
    if "text/html" in content_type:
        # Alguns recursos são páginas HTML, não arquivos. Eles são ignorados.
        return None

    nome = nome_do_arquivo(r, url, indice)
    caminho = destino / nome

    # Evita sobrescrever arquivos de mesmo nome.
    contador = 1
    while caminho.exists():
        caminho = destino / f"{Path(nome).stem}_{contador}{Path(nome).suffix}"
        contador += 1

    with open(caminho, "wb") as arquivo:
        for bloco in r.iter_content(chunk_size=1024 * 256):
            if bloco:
                arquivo.write(bloco)

    return caminho


In [11]:

# Credenciais solicitadas apenas enquanto esta célula está sendo executada.
# NÃO escreva sua senha no notebook.
usuario = input("Número do cartão UFRGS: ").strip()
senha = getpass.getpass("Senha do Moodle/Portal UFRGS: ")

sessao = requests.Session()
sessao.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 Chrome/120 Safari/537.36"
})

fazer_login(sessao, usuario, senha)
print("Login confirmado.")


Login confirmado.


In [6]:

DESTINO.mkdir(parents=True, exist_ok=True)

materiais = coletar_links_de_materiais(sessao, COURSE_URL)
print(f"{len(materiais)} links de materiais encontrados.\n")

for n, link in enumerate(materiais, start=1):
    print(f"{n:03d} — {link}")

if SOMENTE_LISTAR:
    print("\nModo de teste ativo. Altere SOMENTE_LISTAR para False e execute as próximas células para baixar.")


90 links de materiais encontrados.

001 — https://moodle.ufrgs.br/mod/resource/view.php?id=5936612
002 — https://moodle.ufrgs.br/mod/resource/view.php?id=5936614
003 — https://moodle.ufrgs.br/mod/resource/view.php?id=6080894
004 — https://moodle.ufrgs.br/mod/resource/view.php?id=6080957
005 — https://moodle.ufrgs.br/mod/resource/view.php?id=6080958
006 — https://moodle.ufrgs.br/mod/resource/view.php?id=6080959
007 — https://moodle.ufrgs.br/mod/resource/view.php?id=6080960
008 — https://moodle.ufrgs.br/mod/resource/view.php?id=6080961
009 — https://moodle.ufrgs.br/mod/resource/view.php?id=6080962
010 — https://moodle.ufrgs.br/mod/resource/view.php?id=6080963
011 — https://moodle.ufrgs.br/mod/resource/view.php?id=6080964
012 — https://moodle.ufrgs.br/mod/resource/view.php?id=6080965
013 — https://moodle.ufrgs.br/mod/resource/view.php?id=6080967
014 — https://moodle.ufrgs.br/mod/resource/view.php?id=6080968
015 — https://moodle.ufrgs.br/mod/resource/view.php?id=6080969
016 — https://moodl

In [12]:

if SOMENTE_LISTAR:
    print("Download não executado porque SOMENTE_LISTAR=True.")
else:
    baixados, ignorados, erros = [], [], []

    for indice, link in enumerate(tqdm(materiais, desc="Baixando materiais"), start=1):
        try:
            caminho = baixar_material(sessao, link, DESTINO, indice)
            if caminho is None:
                ignorados.append(link)
            else:
                baixados.append(caminho)
        except Exception as e:
            erros.append((link, str(e)))
        time.sleep(PAUSA_ENTRE_DOWNLOADS_S)

    # Relatório local da execução
    relatorio = DESTINO / "_relatorio_download.txt"
    with open(relatorio, "w", encoding="utf-8") as arq:
        arq.write(f"Curso: {COURSE_URL}\n")
        arq.write(f"Arquivos baixados: {len(baixados)}\n")
        arq.write(f"Links ignorados (HTML/páginas): {len(ignorados)}\n")
        arq.write(f"Erros: {len(erros)}\n\n")
        arq.write("ARQUIVOS BAIXADOS\n")
        for item in baixados:
            arq.write(f"- {item.name}\n")
        arq.write("\nERROS\n")
        for link, erro in erros:
            arq.write(f"- {link}\n  {erro}\n")

    print(f"\nConcluído. {len(baixados)} arquivo(s) salvo(s) em:\n{DESTINO}")
    print(f"Relatório: {relatorio}")

    if erros:
        print("\nAlguns links apresentaram erro. Consulte _relatorio_download.txt.")


Baixando materiais:   0%|          | 0/90 [00:00<?, ?it/s]


Concluído. 0 arquivo(s) salvo(s) em:
C:\Users\GSA\Downloads\UFRGS_45959_Metodos_Classicos_Aprendizagem_de_Maquina
Relatório: C:\Users\GSA\Downloads\UFRGS_45959_Metodos_Classicos_Aprendizagem_de_Maquina\_relatorio_download.txt
